<a href="https://colab.research.google.com/github/MabelSSantos/GEDI_start/blob/main/5_GEDI_Pairs_Subregion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DETERMINE THE SUBREGION**

In [ ]:
import geemap
import pandas as pd
import ee
import geopandas as gpd
import math
import time
import seaborn as sns
import matplotlib.pyplot as plt
import os
from google.colab import drive
from pathlib import Path

In [ ]:
ee.Authenticate()
ee.Initialize(project='instant-pivot-489802-p0')

In [ ]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#Input and output paths

#path = Path("/content/drive/MyDrive/GEDI/GEDIdb/GEDI_Footprint_Fire_Pairs_FF_FYs_NDVI.gpkg")
#db_50_path = Path("/content/drive/MyDrive/GEDI/GEDIdb/GEDI_Footprint_50_Fire_Pairs_FF_FYs_NDVI_region.gpkg")
db_25_path = Path("/content/drive/MyDrive/GEDI/GEDIdb/GEDI_Footprint_25_Fire_Pairs_FF_FYs_NDVI_region.gpkg")

AoD_path = Path("/content/drive/MyDrive/GEDI/AoD.geojson")

In [ ]:
# Reading the files

#pairs = gpd.read_file(path)
#pairs_50 = gpd.read_file(db_50_path)
pairs_25 = gpd.read_file(db_25_path)

# Number of pairs
print("Number of pairs:", len(pairs))
#print("Number of pairs in GEDIdb 50:", len(pairs_50))
#print("Number of pairs in GEDI 25:", len(pairs_25))

Number of pairs: 6502


In [ ]:
print(pairs.columns)

Index(['index_1', 'index_2', 'distance_m', 'fire', 'time_1', 'fire_date',
       'time_2', 'fire_frequency', 'fire_years', 'pre_veg_age',
       'post_fire_months', 'agbd_1', 'agbd_2', 'delta_agbd', 'rh_98_1',
       'rh_98_2', 'rh_90_1', 'rh_90_2', 'rh_50_1', 'rh_50_2', 'cover_1',
       'cover_2', 'pai_1', 'pai_2', 'wsci_1', 'wsci_2', 'agbd_pi_lower_1',
       'agbd_pi_upper_1', 'agbd_se_1', 'agbd_pi_lower_2', 'agbd_pi_upper_2',
       'agbd_se_2', 'pair_uid', 'gedi_point_wkt_1', 'gedi_point_wkt_2',
       'pairing_crs', 'geometry_crs_1', 'geometry_crs_2', 'shot_number_1',
       'shot_number_2', 'geometry_1', 'geometry_2', 'pre_img_date',
       'post_img_date', 'ndvi_median_pregeom_at_date1',
       'ndvi_median_postgeom_at_date1', 'ndvi_pre_condition_diff',
       'ndvi_pre_condition_abs_diff', 'geometry'],
      dtype='object')


In [ ]:
print(pairs_25.columns)

Index(['index_1', 'index_2', 'distance_m', 'fire', 'time_1', 'fire_date',
       'time_2', 'season', 'fire_frequency', 'fire_years', 'pre_veg_age',
       'post_fire_months', 'agbd_1', 'agbd_2', 'delta_agbd', 'rh_98_1',
       'rh_98_2', 'rh_90_1', 'rh_90_2', 'rh_50_1', 'rh_50_2', 'cover_1',
       'cover_2', 'pai_1', 'pai_2', 'wsci_1', 'wsci_2', 'agbd_pi_lower_1',
       'agbd_pi_upper_1', 'agbd_se_1', 'agbd_pi_lower_2', 'agbd_pi_upper_2',
       'agbd_se_2', 'img_date', 'ndvi_1', 'ndvi_2', 'delta_ndvi',
       'delta_ndvi_abs', 'pair_uid', 'subregion', 'gedi_point_wkt_1',
       'gedi_point_wkt_2', 'pairing_crs', 'geometry_crs_1', 'geometry_crs_2',
       'shot_number_1', 'shot_number_2', 'geometry_1', 'geometry_2',
       'geometry'],
      dtype='object')


In [ ]:
#If need to change something:

pairs = pairs_25.rename(columns={
    "pre_img_date": "img_date",
    "ndvi_median_pregeom_at_date1": "ndvi_1",
    "ndvi_median_postgeom_at_date1": "ndvi_2",
    "ndvi_pre_condition_diff": "delta_ndvi",
    "ndvi_pre_condition_abs_diff": "delta_ndvi_abs", }

  )

columns_to_keep = [
    'index_1', 'index_2', 'distance_m', 'fire', 'time_1', 'fire_date', 'time_2',
    'fire_frequency','fire_years', 'pre_veg_age', 'post_fire_months',
    'agbd_1', 'agbd_2', 'delta_agbd',
    'rh_98_1', 'rh_98_2', 'rh_90_1', 'rh_90_2', 'rh_50_1', 'rh_50_2',
    'cover_1', 'cover_2', 'pai_1', 'pai_2', 'wsci_1', 'wsci_2',
    'agbd_pi_lower_1', 'agbd_pi_upper_1', 'agbd_se_1',
    'agbd_pi_lower_2', 'agbd_pi_upper_2', 'agbd_se_2',
    'img_date', 'ndvi_1', 'ndvi_2', 'delta_ndvi', 'delta_ndvi_abs',
    'pair_uid',
    'gedi_point_wkt_1', 'gedi_point_wkt_2',
    'pairing_crs', 'geometry_crs_1', 'geometry_crs_2',
    'shot_number_1', 'shot_number_2',
    'geometry_1','geometry_2',
    'geometry'
]

# Keep only the columns that actually exist
columns_to_keep = [col for col in columns_to_keep if col in pairs.columns]

pairs = pairs[columns_to_keep].copy()

print("Updated columns for pairs_25:")
print(pairs.columns)

Updated columns for pairs_50:
Index(['index_1', 'index_2', 'distance_m', 'fire', 'time_1', 'fire_date',
       'time_2', 'fire_frequency', 'fire_years', 'pre_veg_age',
       'post_fire_months', 'agbd_1', 'agbd_2', 'delta_agbd', 'rh_98_1',
       'rh_98_2', 'rh_90_1', 'rh_90_2', 'rh_50_1', 'rh_50_2', 'cover_1',
       'cover_2', 'pai_1', 'pai_2', 'wsci_1', 'wsci_2', 'agbd_pi_lower_1',
       'agbd_pi_upper_1', 'agbd_se_1', 'agbd_pi_lower_2', 'agbd_pi_upper_2',
       'agbd_se_2', 'img_date', 'ndvi_1', 'ndvi_2', 'delta_ndvi',
       'delta_ndvi_abs', 'pair_uid', 'gedi_point_wkt_1', 'gedi_point_wkt_2',
       'pairing_crs', 'geometry_crs_1', 'geometry_crs_2', 'shot_number_1',
       'shot_number_2', 'geometry_1', 'geometry_2', 'geometry'],
      dtype='object')


In [ ]:
def coerce_numeric(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

numeric_cols = [
    'delta_agbd','agbd_1','agbd_2','agbd_se_1','agbd_se_2',
    'delta_ndvi','delta_ndvi_abs','ndvi_1','ndvi_2',
    'post_fire_months','fire_frequency','pre_veg_age',
    'distance_m','rh_98_1','rh_98_2','rh_90_1','rh_90_2',
    'rh_50_1','rh_50_2','cover_1','cover_2','pai_1','pai_2','wsci_1','wsci_2',
    'timedelta_days'  # if you created it earlier
]

# Apply to the frame you’re analyzing
pairs = coerce_numeric(pairs, numeric_cols)

# Remove rows where NDVI_1 or NDVI_2 (or delta_ndvi) are NaN
pairs = pairs.dropna(subset=['ndvi_1', 'ndvi_2', 'delta_ndvi'], how='any')

# Convert post_fire_months to float32
if 'post_fire_months' in pairs.columns:
    pairs['post_fire_months'] = pd.to_numeric(pairs['post_fire_months'], errors='coerce').astype('float32')

# Convert fire_date to datetime64[ms]
if 'fire_date' in pairs.columns:
    # Parse to datetime (keeps tz if present)
    fire_dt = pd.to_datetime(pairs['fire_date'], errors='coerce', utc=True)
    # Convert to UTC (already UTC), drop timezone, and cast to ms resolution
    pairs['fire_date'] = fire_dt.dt.tz_convert('UTC').dt.tz_localize(None).astype('datetime64[ms]')

In [ ]:
#Function to extract season info (dry and wet) and if they are the same

def process_pair_seasons(df):
    # Convert to datetime
    df['time_1'] = pd.to_datetime(df['time_1'])
    df['time_2'] = pd.to_datetime(df['time_2'])

    dry_months = [7, 8, 9, 10, 11]

    def classify_season(dt):
        return "dry" if dt.month in dry_months else "wet"

    df['season_1'] = df['time_1'].apply(classify_season)
    df['season_2'] = df['time_2'].apply(classify_season)

    # Seasons in
    df['season'] = df.apply(
        lambda row: "same" if row['season_1'] == row['season_2'] else "different",
        axis=1
    )

    # Print the number of rows where 'season' is 'same'
    count = len(df[df['season'] == 'same'])
    print(f"Number of rows with 'season' == 'same' for this DataFrame: {count}")

    return df

print("Function 'process_pair_seasons' created successfully.")

Function 'process_pair_seasons' created successfully.


In [ ]:
#To apply the function and get the seasons

# Apply the function to pairs_SR
pairs = process_pair_seasons(pairs)
print("pairs_SR processed with season information.")

# Apply the function to pairs_50
#pairs_50 = process_pair_seasons(pairs_50)
#print("pairs_50 processed with season information.")

# Apply the function to pairs_25
#pairs_25 = process_pair_seasons(pairs_25)
#print("pairs_25 processed with season information.")

Number of rows with 'season' == 'same' for this DataFrame: 3880
pairs_SR processed with season information.


In [ ]:
#Drop the columns again
pairs = pairs.drop(columns=['season_1', 'season_2'])
#pairs_50 = pairs_50.drop(columns=['season_1', 'season_2'])
#pairs_25 = pairs_25.drop(columns=['season_1', 'season_2'])

In [ ]:
AoD = gpd.read_file(AoD_path)

In [ ]:
# Function to get the subregion of each pair -> Original pairs with a new column: subregion

def subregion_by_buffer(
    lines_gdf,                  #Pairs geometries (line) in EPSG:4326 and a 'pairing_crs'
    polygons_gdf,               #Areas geometries in EPSG:4326 -> AoD
    id_col="pair_uid",
    polygon_attr="sub_region",  #Column (from AoD) with the subregion info
    buffer_distance=22.7,       #Buffer distance in meters -> Footpring = 25m (diameter) + Geolocation error = 10.2m (Radius)
    working_crs= "EPSG:6933"    # Using EPSG:6933
):

    # Reproject polygons to the crs
    polys = polygons_gdf.to_crs(working_crs)

    # Buffer each line in its own zonal CRS (UTM)
    buffered_geoms = []

    for idx, row in lines_gdf.iterrows():
        line_geom = row.geometry
        line_crs = row["pairing_crs"]

        # Reproject line to its own CRS
        line_proj = (
            gpd.GeoSeries([line_geom], crs=lines_gdf.crs)
            .to_crs(line_crs)
            .iloc[0]
        )

        # Buffer in that CRS
        buffered = line_proj.buffer(buffer_distance)

        # Reproject buffer to working CRS
        buffered_working = (
            gpd.GeoSeries([buffered], crs=line_crs)
            .to_crs(working_crs)
            .iloc[0]
        )

        buffered_geoms.append(buffered_working)

    # Build buffered GeoDataFrame
    gdf_buffer = gpd.GeoDataFrame(
        lines_gdf[[id_col]].copy(),
        geometry=buffered_geoms,
        crs=working_crs
    )

    # Intersect buffered lines with polygons
    intersections = gpd.overlay(
        gdf_buffer,
        polys,
        how="intersection"
    )

    if intersections.empty:
        lines_gdf["assigned_subregion"] = None
        return lines_gdf

    # Compute intersection area
    intersections["area"] = intersections.geometry.area

    # Select polygon with largest area per line
    best = (
        intersections.sort_values("area", ascending=False)
        .groupby(id_col)
        .first()
    )

    # Merge back into original lines
    result = lines_gdf.merge(
        best[[polygon_attr]],
        on=id_col,
        how="left"
    )

    result = result.rename(columns={polygon_attr: "subregion"})

    return result


In [ ]:
pairs_subregion = subregion_by_buffer(
    pairs,
    AoD,
    id_col="pair_uid",
    polygon_attr="sub_region",
    buffer_distance=22.7,
    working_crs="EPSG:6933"
)

pairs_subregion['subregion'].head()

,subregion
0,central
1,central
2,central
3,central
4,central


In [ ]:
"""
#If need to change something in the columns:

columns_to_keep = [
       'index_1', 'index_2', 'distance_m', 'time_1','time_2', 'season',
       'fire_date', 'post_fire_months', 'agbd_1', 'agbd_2',
       'delta_agbd', 'geometry_1', 'geometry_2', 'pairing_crs', 'pair_uid',
       'fire_frequency', 'fire_years', 'pre_veg_age', 'img_date', 'ndvi_1',
       'ndvi_2', 'delta_ndvi', 'delta_ndvi_abs', 'subregion','geometry'
       ]

# Keep only the columns that actually exist
columns_to_keep = [col for col in columns_to_keep if col in pairs_SR_subregion.columns]

pairs_subregion = pairs_subregion[columns_to_keep].copy()

print("Updated columns for pairs_SR:")
print(pairs_SR_subregion.columns)

Updated columns for pairs_SR:
Index(['index_1', 'index_2', 'distance_m', 'time_1', 'time_2', 'season',
       'fire_date', 'post_fire_months', 'agbd_1', 'agbd_2', 'delta_agbd',
       'geometry_1', 'geometry_2', 'pairing_crs', 'pair_uid', 'fire_frequency',
       'fire_years', 'pre_veg_age', 'img_date', 'ndvi_1', 'ndvi_2',
       'delta_ndvi', 'delta_ndvi_abs', 'subregion', 'geometry'],
      dtype='object')


In [ ]:
"""

#If need to update the files
# Save pairs_SR to its original path
pairs_SR.to_file(SR_path, driver='GPKG')
print(f"pairs_SR saved to: {SR_path}")

# Save pairs_50 to its original path
pairs_50.to_file(db_50_path, driver='GPKG')
print(f"pairs_50 saved to: {db_50_path}")

# Save pairs_25 to its original path
pairs_25.to_file(db_25_path, driver='GPKG')
print(f"pairs_25 saved to: {db_25_path}")

pairs_SR saved to: /content/drive/MyDrive/GEDI/SlideRule/GEDI_Footprint_SR_Fire_Pairs_FF_FYs_NDVI.gpkg
pairs_50 saved to: /content/drive/MyDrive/GEDI/GEDIdb/GEDI_Footprint_50_Fire_Pairs_FF_FYs_NDVI.gpkg
pairs_25 saved to: /content/drive/MyDrive/GEDI/GEDIdb/GEDI_Footprint_25_Fire_Pairs_FF_FYs_NDVI.gpkg


In [ ]:
print(f"Number of pairs in each region in SR: {pairs_subregion['subregion'].value_counts()}")
#print(f"Number of pairs in each region in GEDIdb 50: {pairs_50_subregion['subregion'].value_counts()}")
#print(f"Number of pairs in each region in GEDIdb 25: {pairs_25_subregion['subregion'].value_counts()}")

Number of pairs in each region in SR: subregion
central    3126
west       3032
east        344
Name: count, dtype: int64


In [ ]:
print(f"Number of pairs with = and =/ seasons in SR: {pairs_subregion['season'].value_counts()}")
#print(f"Number of pairs with = and =/ seasons in GEDIdb 50: {pairs_50_subregion['season'].value_counts()}")
#print(f"Number of pairs with = and =/ seasons in GEDIdb 25: {pairs_25_subregion['season'].value_counts()}")

Number of pairs with = and =/ seasons in SR: season
same         3880
different    2622
Name: count, dtype: int64


In [ ]:
desired_order = [
    'index_1', 'index_2', 'distance_m', 'fire', 'time_1', 'fire_date',
    'time_2', 'season', 'fire_frequency', 'fire_years', 'pre_veg_age',
    'post_fire_months', 'agbd_1', 'agbd_2', 'delta_agbd', 'rh_98_1',
    'rh_98_2', 'rh_90_1', 'rh_90_2', 'rh_50_1', 'rh_50_2', 'cover_1',
    'cover_2', 'pai_1', 'pai_2', 'wsci_1', 'wsci_2', 'agbd_pi_lower_1',
    'agbd_pi_upper_1', 'agbd_se_1', 'agbd_pi_lower_2', 'agbd_pi_upper_2',
    'agbd_se_2', 'img_date', 'ndvi_1', 'ndvi_2', 'delta_ndvi',
    'delta_ndvi_abs', 'pair_uid', 'subregion', 'gedi_point_wkt_1',
    'gedi_point_wkt_2', 'pairing_crs', 'geometry_crs_1', 'geometry_crs_2',
    'shot_number_1', 'shot_number_2', 'geometry_1', 'geometry_2', 'geometry'
]

#pairs_50_subregion = pairs_50_subregion[desired_order]
pairs_25_subregion = pairs_25_subregion[desired_order]

In [ ]:
def add_suffix(path, suffix="_region"):
    base, ext = os.path.splitext(path)
    return f"{base}{suffix}{ext}"

# Build new paths
#path_region = add_suffix(path)
#db_50_path_region = add_suffix(db_50_path)
db_25_path_region = add_suffix(db_25_path)

# Save files
#pairs_subregion.to_file(path_region, driver='GPKG')
#print(f"pairs_SR saved to: {path_region}")

#pairs_50_subregion.to_file(db_50_path_region, driver='GPKG')
#print(f"pairs_50 saved to: {db_50_path_region}")

pairs_25_subregion.to_file(db_25_path_region, driver='GPKG')
print(f"pairs_25 saved to: {db_25_path_region}")

pairs_SR saved to: /content/drive/MyDrive/GEDI/GEDIdb/GEDI_Footprint_Fire_Pairs_FF_FYs_NDVI_region.gpkg
